# Data Analysis: Sales Data
Generated by autonomous data-analyst agent.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pathlib, warnings
warnings.filterwarnings('ignore')

# Settings
sns.set_palette('tab10')
plt.rcParams['figure.dpi'] = 100


In [ ]:

data_path = '/workspace/repo/sales_data.csv'
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])
print(f'Loaded: {df.shape}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
df.head()


## Statistical Summary

In [ ]:

def describe_column(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return {
        'count': len(series),
        'null_pct': round(series.isna().mean() * 100, 2),
        'mean': round(series.mean(), 4),
        'median': round(series.median(), 4),
        'std': round(series.std(), 4),
        'iqr': round(iqr, 4),
        'p5': round(series.quantile(0.05), 4),
        'p95': round(series.quantile(0.95), 4),
        'outlier_count': int(((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum()),
    }

numeric_cols = df.select_dtypes(include='number').columns
summary = pd.DataFrame({col: describe_column(df[col].dropna()) for col in numeric_cols}).T
print(summary.to_string())


## Distributions

In [ ]:

numeric_cols = df.select_dtypes(include='number').columns.tolist()
cols_to_plot = numeric_cols[:6]
n = len(cols_to_plot)
if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, cols_to_plot):
        ax.hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white')
        ax.axvline(df[col].median(), color='orange', linestyle='--', label=f'Median ({df[col].median():.2f})')
        ax.axvline(df[col].mean(), color='red', linestyle=':', label=f'Mean ({df[col].mean():.2f})')
        ax.set_title(f'Distribution: {col}', fontsize=11, fontweight='bold')
        ax.set_xlabel(col)
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved distributions.png')


## Correlation Heatmap

In [ ]:

num_df = df.select_dtypes(include='number')
if num_df.shape[1] >= 2:
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(num_df.corr(), ax=ax, cmap='coolwarm', center=0, annot=True, fmt='.2f', square=True)
    ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved correlation_heatmap.png')


## Outlier Analysis (IQR method)

In [ ]:

numeric_cols = df.select_dtypes(include='number').columns
outlier_report = {}
for col in numeric_cols:
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = df[(df[col] < lo) | (df[col] > hi)]
    outlier_report[col] = {
        'count': len(outliers),
        'pct': round(len(outliers) / len(df) * 100, 2),
        'low_bound': round(lo, 4),
        'high_bound': round(hi, 4)
    }
outlier_df = pd.DataFrame(outlier_report).T
print(outlier_df.to_string())


## Normality Tests (Shapiro-Wilk, n ≤ 5000)

In [ ]:

normality_results = {}
for col in df.select_dtypes(include='number').columns:
    s = df[col].dropna()
    sample = s.sample(min(len(s), 500), random_state=42)
    stat, p = stats.shapiro(sample)
    normality_results[col] = {'statistic': round(stat, 4), 'p_value': round(p, 6), 'normal': p > 0.05}
norm_df = pd.DataFrame(normality_results).T
print(norm_df.to_string())


## Time Series: Daily Sales

In [ ]:

daily_sales = df.groupby('date')['sales_amount'].agg(['sum', 'mean', 'count']).reset_index()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily_sales['date'], daily_sales['sum'], marker='o', linestyle='-', color='steelblue', label='Total Sales')
ax.set_title('Daily Total Sales Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Sales Amount')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('time_series_daily_sales.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved time_series_daily_sales.png')


## Sales by Region

In [ ]:

region_stats = df.groupby('region')['sales_amount'].agg(['count', 'mean', 'median', 'std']).reset_index()
print(region_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='region', y='sales_amount', palette='Set2', ax=ax)
ax.set_title('Sales Amount Distribution by Region', fontsize=14, fontweight='bold')
ax.set_xlabel('Region')
ax.set_ylabel('Sales Amount')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('sales_by_region.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved sales_by_region.png')


## Sales by Product

In [ ]:

product_stats = df.groupby('product_id')['sales_amount'].agg(['count', 'mean', 'median', 'std']).reset_index()
print(product_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='product_id', y='sales_amount', palette='viridis', ax=ax)
ax.set_title('Sales Amount Distribution by Product ID', fontsize=14, fontweight='bold')
ax.set_xlabel('Product ID')
ax.set_ylabel('Sales Amount')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('sales_by_product.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved sales_by_product.png')
